# 1) Setup & Imports

Dans cette section, on installe les packages nécessaires et on importe toutes les bibliothèques utilisées dans le projet :
- **Data** : pandas, numpy, yfinance, requests
- **Indicators** : pandas_ta
- **Models** : KMeans (clustering), RollingOLS (betas), LSTM (TensorFlow)
- **Portfolio optimization** : PyPortfolioOpt
- **Visualisation** : matplotlib


In [5]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
import pandas_ta as ta
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from statsmodels.regression.rolling import RollingOLS
import statsmodels.api as sm

from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models

import tensorflow as tf
from tensorflow.keras import layers, callbacks, models

import datetime as dt
from io import StringIO


# 2) Paramètres du projet

On fixe ici tous les hyperparamètres et réglages globaux :

- **Période d’étude** : 8 ans de données jusqu’au 27/09/2023  
- **Clustering** : KMeans avec `N_CLUSTERS=4`  
- **Sélection** : on garde les top actions dans les rôles *Gagnants dynamiques* et *Stables de qualité*  
- **Backtest / portefeuille** : fenêtre daily de 12 mois + contrainte de poids max par action  
- **Modèle prédictif** : LSTM sur séquences de 12 mois pour prédire le retour du mois suivant  
- **Reproductibilité** : seed fixé pour obtenir les mêmes résultats à chaque exécution


In [6]:
end_date = "2023-09-27"
start_date = (pd.to_datetime(end_date) - pd.DateOffset(years=8)).strftime("%Y-%m-%d")

N_CLUSTERS = 4
TOP_FRAC = 0.50         # sélection top 50% dans Winners & Stables
MIN_NAMES = 10          # minimum d’actions par rôle
MAX_NAMES = 60          # optionnel : plafond

LOOKBACK_MONTHS_DAILY = 12  # pour covariance et perf (fenêtre daily)
MAX_WEIGHT = 0.10

# LSTM
SEQ_LEN = 12            # 12 mois d'historique pour prédire le mois suivant
LSTM_EPOCHS = 30
BATCH_SIZE = 256
LR = 1e-3
RANDOM_SEED = 0

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# 3) Collecte des données (S&P 500 + prix journaliers)

On récupère :
1. La liste des entreprises du **S&P 500** depuis Wikipedia (tickers).
2. Les prix journaliers OHLCV depuis **Yahoo Finance** via `yfinance`.

Le résultat est un DataFrame au format **MultiIndex** `(Date, Ticker)` avec les colonnes :
`Open, High, Low, Close, Adj Close, Volume`.

> Remarque : certains tickers contiennent un point (ex: `BRK.B`). Sur Yahoo Finance, on remplace `.` par `-` (ex: `BRK-B`).


In [ ]:
# ============================================================
# DOWNLOAD S&P500 TICKERS (Wikipedia)
# ============================================================
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

sp500 = pd.read_html(StringIO(response.text))[0]

sp500["Symbol"] = sp500["Symbol"].str.replace(".", "-", regex=False)
symbols = sp500["Symbol"].unique().tolist()

print("Nb tickers (wiki):", len(symbols))

# ============================================================
# DOWNLOAD DAILY PRICES
# ============================================================
data = yf.download(symbols, start=start_date, end=end_date, auto_adjust=False, progress=False).stack()
df = data.copy()
df.columns.name = None

print("\nColonnes:", df.columns.tolist())
print("Nb tickers réellement téléchargés:", df.index.get_level_values("Ticker").nunique())
print("Période:", df.index.get_level_values("Date").min().date(), "→", df.index.get_level_values("Date").max().date())

display(df)

Nb tickers (wiki): 503


$Q: possibly delisted; no price data found  (1d 2015-09-27 -> 2023-09-27) (Yahoo error = "Data doesn't exist for startDate = 1443326400, endDate = 1695787200")
$VLTO: possibly delisted; no price data found  (1d 2015-09-27 -> 2023-09-27) (Yahoo error = "Data doesn't exist for startDate = 1443326400, endDate = 1695787200")
$SOLV: possibly delisted; no price data found  (1d 2015-09-27 -> 2023-09-27) (Yahoo error = "Data doesn't exist for startDate = 1443326400, endDate = 1695787200")
$SNDK: possibly delisted; no price data found  (1d 2015-09-27 -> 2023-09-27) (Yahoo error = "Data doesn't exist for startDate = 1443326400, endDate = 1695787200")
$GEV: possibly delisted; no price data found  (1d 2015-09-27 -> 2023-09-27) (Yahoo error = "Data doesn't exist for startDate = 1443326400, endDate = 1695787200")

5 Failed downloads:
['Q', 'VLTO', 'SOLV', 'SNDK', 'GEV']: possibly delisted; no price data found  (1d 2015-09-27 -> 2023-09-27) (Yahoo error = "Data doesn't exist for startDate = 144332640


Colonnes: ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
Nb tickers réellement téléchargés: 498
Période: 2015-09-28 → 2023-09-26


Adj Close       Close        High         Low        Open  \
Date       Ticker                                                               
2015-09-28 A        30.656668   33.369999   34.270000   33.320000   34.130001   
           AAPL     25.220074   28.110001   28.642500   28.110001   28.462500   
           ABBV     34.048641   52.500000   55.430000   52.340000   55.389999   
           ABT      32.004959   39.060001   40.740002   39.000000   40.740002   
           ACGL     22.992727   24.180000   24.243334   23.966667   24.056667   
...                       ...         ...         ...         ...         ...   
2023-09-26 XYZ      44.810001   44.810001   45.740002   44.560001   44.939999   
           YUM     118.174828  124.010002  124.739998  123.449997  124.239998   
           ZBH     109.653664  112.459999  117.110001  112.419998  116.769997   
           ZBRA    223.960007  223.960007  226.649994  222.580002  225.970001   
           ZTS     171.692566  176.869995  178.449997  176.270004  176.580002   

                        Volume  
Date       Ticker               
2015-09-28 A         2615400.0  
           AAPL    208436000.0  
           ABBV     15123000.0  
           ABT      17219300.0  
           ACGL      1946700.0  
...                        ...  
2023-09-26 XYZ      10168200.0  
           YUM       1500600.0  
           ZBH       3610500.0  
           ZBRA       355400.0  
           ZTS       1463200.0  

[974542 rows x 6 columns]

# 4) Feature engineering (données journalières)

On calcule des indicateurs journaliers utiles pour :
- **Volatilité** : `GK` (Garman–Klass) ≈ volatilité dans la même journée réalisée basée sur OHLC
- **Liquidité** : `dollar_volume` (Volume × Adj Close)
- **Momentum** : `RSI(20)`
- **Risque / volatilité** : `ATR(14)` (Average True Range)

Ces features seront ensuite agrégées au **niveau mensuel** pour faire le clustering et les prédictions.


In [8]:
# ============================================================
# 3) FEATURES DAILY
# ============================================================

# Suppression des tickers vides pour éviter les erreurs dans pandas_ta
df = df.dropna(subset=['High', 'Low', 'Close'])

df["GK"] = np.sqrt(np.maximum(
    0,
    0.5 * (np.log(df["High"] / df["Low"]))**2
    - (2 * np.log(2) - 1) * (np.log(df["Close"] / df["Open"]))**2
))

df["dollar_volume"] = (df["Volume"] * df["Adj Close"]) / 1e6
df["rsi"] = df.groupby("Ticker")["Adj Close"].transform(lambda x: ta.rsi(close=x, length=20))

def compute_atr(stock_data):
    # Sécurité supplémentaire : vérifier si assez de données pour la période 14
    if len(stock_data) < 14:
        return pd.Series(np.nan, index=stock_data.index)
    return ta.atr(high=stock_data["High"], low=stock_data["Low"], close=stock_data["Close"], length=14)

df["atr"] = df.groupby(level="Ticker", group_keys=False).apply(compute_atr)

display(df.dropna())

Adj Close       Close        High         Low        Open  \
Date       Ticker                                                               
2015-10-15 A        32.686958   35.580002   35.680000   34.790001   35.119999   
           AAPL     25.089977   27.965000   28.025000   27.622499   27.732500   
           ABBV     36.745380   56.139999   56.200001   53.849998   54.320000   
           ABT      34.506622   41.869999   42.029999   40.910000   41.070000   
           ACGL     24.330322   25.586666   25.606667   25.240000   25.253332   
...                       ...         ...         ...         ...         ...   
2023-09-26 XYZ      44.810001   44.810001   45.740002   44.560001   44.939999   
           YUM     118.174828  124.010002  124.739998  123.449997  124.239998   
           ZBH     109.653664  112.459999  117.110001  112.419998  116.769997   
           ZBRA    223.960007  223.960007  226.649994  222.580002  225.970001   
           ZTS     171.692566  176.869995  178.449997  176.270004  176.580002   

                        Volume        GK  dollar_volume        rsi       atr  
Date       Ticker                                                             
2015-10-15 A         1433000.0  0.015926      46.840411  79.860197  0.818572  
           AAPL    150694000.0  0.008815    3780.909034  12.585861  0.646964  
           ABBV      7781600.0  0.022197     285.937852  75.787717  1.857857  
           ABT       7822000.0  0.014865     269.910800  82.004771  1.077857  
           ACGL      1433100.0  0.006130      34.867785  88.097608  0.380714  
...                        ...       ...            ...        ...       ...  
2023-09-26 XYZ      10168200.0  0.018393     455.637056  25.939817  1.908641  
           YUM       1500600.0  0.007260     177.333146  36.057155  1.860258  
           ZBH       3610500.0  0.016996     395.904553  31.893264  2.360510  
           ZBRA       355400.0  0.011547      79.595386  29.494977  6.949335  
           ZTS       1463200.0  0.008631     251.220562  42.623429  3.616068  

[968068 rows x 10 columns]

# 5) Passage en données mensuelles (resample)

Pour le clustering et la stratégie (rééquilibrage mensuel), on agrège les données journalières en **format mensuel**.

Règles d’agrégation utilisées :
- **Open** : premier jour du mois
- **High** : maximum du mois
- **Low** : minimum du mois
- **Close / Adj Close** : dernier jour du mois
- **Volume / dollar_volume** : somme mensuelle (activité totale)
- **GK** : moyenne mensuelle de la volatilité journalière
- **RSI / ATR** : valeur du dernier jour du mois (cohérent avec un signal fin de mois)

On ajoute ensuite `atr_pct = ATR / AdjClose`, une mesure de risque relative.


In [9]:
# ============================================================
# 4) RESAMPLE MONTHLY
# ============================================================
df1 = df.sort_index().reset_index().set_index("Date")

df_monthly = (
    df1.groupby("Ticker")
       .resample("ME")
       .agg({
           "Open": "first",
           "High": "max",
           "Low": "min",
           "Close": "last",
           "Adj Close": "last",
           "Volume": "sum",
           "dollar_volume": "sum",
           "GK": "mean",
           "rsi": "last",
           "atr": "last"
       })
)

df1 = (df_monthly.reset_index()
       .set_index(["Date", "Ticker"])
       .sort_index()
       .dropna())

df1["atr_pct"] = df1["atr"] / df1["Adj Close"]

print("Monthly dataset created:", df1.shape)
display(df1)

Monthly dataset created: (46474, 11)


Open        High         Low       Close   Adj Close  \
Date       Ticker                                                               
2015-10-31 A        34.320000   38.099998   33.119999   37.759998   34.689705   
           AAPL     27.267500   30.305000   26.827499   29.875000   26.803621   
           ABBV     54.680000   60.730000   45.450001   59.549999   38.977318   
           ABT      40.349998   45.340000   39.279999   44.799999   36.921345   
           ACGL     24.613333   26.270000   24.126667   24.963333   23.737597   
...                       ...         ...         ...         ...         ...   
2023-09-30 XYZ      58.389999   59.720001   44.330002   44.810001   44.810001   
           YUM     129.929993  132.350006  123.449997  124.010002  118.174828   
           ZBH     119.610001  123.769997  112.419998  112.459999  109.653664   
           ZBRA    276.959991  278.760010  222.580002  223.960007  223.960007   
           ZTS     192.970001  193.350006  174.970001  176.869995  171.692566   

                         Volume  dollar_volume        GK        rsi       atr  \
Date       Ticker                                                               
2015-10-31 A       5.299330e+07    1751.930948  0.011861  80.302119  0.737763   
           AAPL    4.450048e+09  113451.573615  0.011658  34.131053  0.669370   
           ABBV    2.669152e+08    9347.820036  0.025533  67.417599  2.642857   
           ABT     1.734921e+08    6019.552137  0.012659  81.279843  0.909274   
           ACGL    3.011820e+07     725.476780  0.008494  65.600917  0.394141   
...                         ...            ...       ...        ...       ...   
2023-09-30 XYZ     1.936574e+08    9888.450048  0.019015  25.939817  1.908641   
           YUM     2.372670e+07    2877.820222  0.008523  36.057155  1.860258   
           ZBH     2.746270e+07    3199.015053  0.010700  31.893264  2.360510   
           ZBRA    7.240600e+06    1798.274678  0.013286  29.494977  6.949335   
           ZTS     2.694490e+07    4781.386400  0.011530  42.623429  3.616068   

                    atr_pct  
Date       Ticker            
2015-10-31 A       0.021267  
           AAPL    0.024973  
           ABBV    0.067805  
           ABT     0.024627  
           ACGL    0.016604  
...                     ...  
2023-09-30 XYZ     0.042594  
           YUM     0.015742  
           ZBH     0.021527  
           ZBRA    0.031029  
           ZTS     0.021061  

[46474 rows x 11 columns]

# 6) Filtre de liquidité + calcul des rendements mensuels

## 6.1 Filtre de liquidité (Top 150)
Pour éviter les actions peu liquides (spread élevé, prix bruités, impossible à trader), on conserve chaque mois uniquement les **actions les plus liquides**.

On utilise `dollar_volume` (Volume × Prix) agrégé au mois, puis on calcule une moyenne glissante sur **5 ans** (60 mois) afin de stabiliser la mesure de liquidité.

Ensuite, à chaque fin de mois, on garde les **150** actions avec le plus haut dollar_volume.

## 6.2 Rendements multi-horizons
On calcule des rendements mensuels sur plusieurs horizons : 1, 2, 3, 6, 9, 12 mois.
La formule utilisée correspond à un rendement “moyen par mois” (normalisé par l’horizon).
Ces features serviront au clustering et à la modélisation.


In [10]:
# ============================================================
# 5) LIQUIDITY FILTER TOP 150 (rolling 5 years monthly mean)
# ============================================================
df1["dollar_volume"] = df1["dollar_volume"].unstack("Ticker").rolling(5*12, min_periods=12).mean().stack()
df1["classement"] = df1.groupby("Date")["dollar_volume"].rank(ascending=False)
df1 = df1[df1["classement"] < 150].drop(["dollar_volume", "classement"], axis=1)

# ============================================================
# 6) MONTHLY RETURNS (1,2,3,6,9,12m annualized-ish as you did)
# ============================================================
def calculate_returns(g):
    for i in [1,2,3,6,9,12]:
        g[f"return_{i}m"] = (g["Adj Close"].pct_change(i).add(1).pow(1/i).sub(1))
    return g

df1 = df1.groupby(level=1, group_keys=False).apply(calculate_returns).dropna()

display(df1)

Open        High         Low       Close   Adj Close  \
Date       Ticker                                                               
2017-09-30 AAPL     41.200001   41.235001   37.290001   38.529999   35.930733   
           ABBV     75.589996   90.949997   74.730003   88.860001   62.180595   
           ABT      51.029999   54.799999   50.639999   53.360001   45.904259   
           ACN     130.949997  138.699997  129.100006  135.070007  118.876755   
           ADBE    155.759995  157.889999  143.949997  149.179993  149.179993   
...                       ...         ...         ...         ...         ...   
2023-09-30 WDAY    246.539993  252.720001  228.559998  229.240005  229.240005   
           WFC      41.549999   43.740002   40.299999   40.650002   38.230572   
           WMT      54.290001   55.283333   53.303333   54.166668   52.713261   
           XOM     112.199997  118.839996  112.199997  116.410004  106.888481   
           XYZ      58.389999   59.720001   44.330002   44.810001   44.810001   

                         Volume        GK        rsi       atr   atr_pct  \
Date       Ticker                                                          
2017-09-30 AAPL    2.721496e+09  0.009763  45.768757  0.651081  0.018120   
           ABBV    1.922700e+08  0.014409  70.568558  1.967963  0.031649   
           ABT     1.250395e+08  0.007494  63.664994  0.811059  0.017669   
           ACN     5.082070e+07  0.007929  56.250081  1.871653  0.015744   
           ADBE    5.508500e+07  0.008799  47.932484  2.460673  0.016495   
...                         ...       ...        ...       ...       ...   
2023-09-30 WDAY    2.205610e+07  0.010539  43.976804  5.007694  0.021845   
           WFC     2.961534e+08  0.012576  40.920269  0.921229  0.024097   
           WMT     2.850591e+08  0.006472  54.722578  0.579568  0.010995   
           XOM     2.609934e+08  0.010058  59.440190  2.103267  0.019677   
           XYZ     1.936574e+08  0.019015  25.939817  1.908641  0.042594   

                   return_1m  return_2m  return_3m  return_6m  return_9m  \
Date       Ticker                                                          
2017-09-30 AAPL    -0.060244   0.019956   0.024187   0.013139   0.033672   
           ABBV     0.180080   0.127414   0.073326   0.056376   0.042995   
           ABT      0.047507   0.041631   0.033465   0.033071   0.039285   
           ACN      0.032961   0.023971   0.029801   0.021839   0.017126   
           ADBE    -0.038541   0.009140   0.017918   0.023031   0.042073   
...                      ...        ...        ...        ...        ...   
2023-09-30 WDAY    -0.062413  -0.016777   0.004919   0.017531   0.035597   
           WFC     -0.015500  -0.057917  -0.013554   0.016712   0.000702   
           WMT     -0.000676   0.010014   0.012354   0.017574   0.016553   
           XOM      0.046947   0.046139   0.030496   0.012838   0.008747   
           XYZ     -0.222723  -0.254052  -0.123607  -0.068630  -0.036876   

                   return_12m  
Date       Ticker              
2017-09-30 AAPL      0.027658  
           ABBV      0.032249  
           ABT       0.021639  
           ACN       0.010139  
           ADBE      0.026857  
...                       ...  
2023-09-30 WDAY      0.034709  
           WFC       0.003255  
           WMT       0.020256  
           XOM       0.027037  
           XYZ      -0.016915  

[10462 rows x 16 columns]

# 7) Facteurs Fama-French + betas glissants (RollingOLS)

Objectif :
- Ajouter une dimension “style / risque” aux actions via les facteurs Fama-French
- Estimer des **expositions (betas)** qui évoluent dans le temps

Étapes :
1. Charger les facteurs Fama-French depuis `fama_french.csv` (format quotidien)
2. Convertir en format mensuel
3. Joindre `return_1m` de chaque action
4. Estimer des betas glissants avec une régression OLS sur une fenêtre de 24 mois


In [12]:
# ============================================================
# 7) FAMA-FRENCH + ROLLING BETAS
#    (Garde ta logique "RF négligeable": on drop RF)
# ============================================================
fama_french = pd.read_csv("docs/data/fama_french.csv", skiprows=4).drop("RF", axis=1, errors="ignore")
fama_french.rename(columns={"Unnamed: 0": "Date"}, inplace=True)
fama_french["Date"] = pd.to_datetime(fama_french["Date"], format="%Y%m%d")
fama_french = fama_french.set_index("Date").sort_index()
fama_french = fama_french / 100
fama_french = fama_french[fama_french.index >= "2017-01-01"].resample("ME").mean()

# join return_1m to factors per ticker -> multiindex (Date,Ticker)
ff = fama_french.join(df1["return_1m"]).sort_index()

# filter tickers with min history
counts = ff.groupby(level=1).size()
valid = counts[counts >= 10].index
ff = ff[ff.index.get_level_values("Ticker").isin(valid)]

# rolling OLS per ticker
betas = (
    ff.groupby(level=1, group_keys=False)
      .apply(lambda x: RollingOLS(
          endog=x["return_1m"],
          exog=sm.add_constant(x.drop("return_1m", axis=1)),
          window=min(24, x.shape[0]),
          min_nobs=len(x.columns)+1
      ).fit(params_only=True).params.drop("const", axis=1))
)

factors = [c for c in fama_french.columns]

# join betas shifted -> avoid leakage
df1 = df1.join(betas.groupby("Ticker").shift())

# fill remaining betas with mean per ticker
df1.loc[:, factors] = df1.groupby("Ticker", group_keys=False)[factors].apply(lambda x: x.fillna(x.mean()))
df1 = df1.dropna()

print("df1 after adding rolling betas:", df1.shape)

df1 after adding rolling betas: (10218, 21)


# 8) Clustering mensuel (KMeans) — avec standardisation

Objectif : regrouper les actions chaque mois en “familles” similaires selon leurs caractéristiques (momentum, volatilité, exposition aux facteurs, etc.).

Points importants :
- On fait le clustering **mois par mois** (car le marché change dans le temps)
- On standardise les features **au sein de chaque mois** (`StandardScaler`) :
  - sinon une feature (ex: volatilité) pourrait dominer le clustering uniquement à cause de son échelle


In [13]:
# ============================================================
# 8) BUILD FEATURE SET FOR CLUSTERING
# ============================================================
drop_cols = ["Volume","Open","High","Low","Close","Adj Close","atr"]
features = [c for c in df1.columns if c not in drop_cols]
df_feat = df1[features].copy()

# ============================================================
# CLUSTERING MONTH-BY-MONTH WITH STANDARDIZATION (FIX)
# ============================================================
def get_clusters_one_month(g, n_clusters=N_CLUSTERS, random_state=0):
    X = g.values
    Xs = StandardScaler().fit_transform(X)  # standardisation within the month
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labels = km.fit_predict(Xs)
    out = g.copy()
    out["cluster"] = labels
    return out

df_clustered = df_feat.dropna().groupby("Date", group_keys=False).apply(get_clusters_one_month)

print("Clustered dataset:", df_clustered.shape)
display(df_clustered.head())

Clustered dataset: (10218, 15)


GK        rsi   atr_pct  return_1m  return_2m  \
Date       Ticker                                                        
2017-09-30 AAPL    0.009763  45.768757  0.018120  -0.060244   0.019956   
           ABBV    0.014409  70.568558  0.031649   0.180080   0.127414   
           ABT     0.007494  63.664994  0.017669   0.047507   0.041631   
           ACN     0.007929  56.250081  0.015744   0.032961   0.023971   
           ADBE    0.008799  47.932484  0.016495  -0.038541   0.009140   

                   return_3m  return_6m  return_9m  return_12m     Mkt-RF  \
Date       Ticker                                                           
2017-09-30 AAPL     0.024187   0.013139   0.033672    0.027658  26.119671   
           ABBV     0.073326   0.056376   0.042995    0.032249  10.782336   
           ABT      0.033465   0.033071   0.039285    0.021639  18.250758   
           ACN      0.029801   0.021839   0.017126    0.010139  25.163335   
           ADBE     0.017918   0.023031   0.042073    0.026857  23.447905   

                        SMB        HML        RMW        CMA  cluster  
Date       Ticker                                                      
2017-09-30 AAPL   -3.766800 -11.945149  18.515471   7.416116        2  
           ABBV    6.575230   0.132594   1.436639   2.463194        3  
           ABT    -3.315076 -10.978257   5.828022  19.639424        2  
           ACN    -3.063859  -5.825993   6.367886   1.512977        2  
           ADBE   -6.014644  -3.127997  -3.335185 -15.626848        2

# 9) Interpréter les clusters : attribution de rôles + scores

Le clustering (KMeans) donne des labels `0,1,2,3` qui ne veulent rien dire directement.
Pour rendre le modèle interprétable, on attribue à chaque cluster un **rôle** à partir de profils moyens :

- **Gagnants dynamiques** : forts rendements passés (momentum)
- **Stables de qualité** : profil “qualité” (ex : RMW élevé) et risque modéré
- **Sous pression** : mauvais rendement + volatilité élevée
- **Volatils neutres** : le(s) cluster(s) restant(s)

Méthode :
1. Pour un mois donné, on calcule la moyenne des features par cluster
2. On construit 4 scores (Trend, Risk, Quality, Stress)
3. On assigne un rôle unique à chaque cluster (mapping sans conflit)
4. On répète pour tous les mois


In [14]:
# ============================================================
# 9) ROLE + SCORES (as you did)
# ============================================================
def add_cluster_roles(df_month):
    g = df_month.copy()
    feats = [c for c in g.columns if c != "cluster"]
    prof = g.groupby("cluster")[feats].mean(numeric_only=True)

    trend = 0.45*prof["return_12m"] + 0.35*prof["return_6m"] + 0.20*prof["return_3m"] + 0.05*(prof["rsi"]/100)
    risk  = 1.00*prof["GK"] + 1.00*prof["atr_pct"]
    quality = 1.00*prof["RMW"] - 0.30*prof["GK"]
    stress  = -1.00*prof["return_12m"] + 0.80*prof["GK"] + 0.20*((50 - prof["rsi"])/50)

    scores = pd.DataFrame({
        "TrendScore": trend,
        "RiskScore": risk,
        "QualityScore": quality,
        "StressScore": stress
    })

    role_map = {}
    cl_win = scores["TrendScore"].idxmax()
    role_map[cl_win] = "Gagnants dynamiques"

    remaining = scores.index.difference(role_map.keys())
    cl_stress = scores.loc[remaining, "StressScore"].idxmax()
    role_map[cl_stress] = "Sous pression"

    remaining = scores.index.difference(role_map.keys())
    stable_metric = scores.loc[remaining, "QualityScore"] - 0.5*scores.loc[remaining, "RiskScore"]
    cl_stable = stable_metric.idxmax()
    role_map[cl_stable] = "Stables de qualité"

    remaining = scores.index.difference(role_map.keys())
    for cl in remaining:
        role_map[cl] = "Volatils neutres"

    g["role"] = g["cluster"].map(role_map)
    g = g.join(scores, on="cluster")
    return g, role_map

def apply_roles_all_months(df_in):
    out = []
    role_maps = {}
    for d in df_in.index.get_level_values("Date").unique().sort_values():
        g = df_in.xs(d, level=0).copy()
        g2, rmap = add_cluster_roles(g)
        g2["Date"] = d
        out.append(g2)
        role_maps[d] = rmap
    df_roles = (pd.concat(out)
                .reset_index()
                .set_index(["Date","Ticker"])
                .sort_index())
    return df_roles, role_maps

df1_roles, role_maps = apply_roles_all_months(df_clustered)

print("df1_roles:", df1_roles.shape)
display(df1_roles.head())

df1_roles: (10218, 20)


GK        rsi   atr_pct  return_1m  return_2m  \
Date       Ticker                                                        
2017-09-30 AAPL    0.009763  45.768757  0.018120  -0.060244   0.019956   
           ABBV    0.014409  70.568558  0.031649   0.180080   0.127414   
           ABT     0.007494  63.664994  0.017669   0.047507   0.041631   
           ACN     0.007929  56.250081  0.015744   0.032961   0.023971   
           ADBE    0.008799  47.932484  0.016495  -0.038541   0.009140   

                   return_3m  return_6m  return_9m  return_12m     Mkt-RF  \
Date       Ticker                                                           
2017-09-30 AAPL     0.024187   0.013139   0.033672    0.027658  26.119671   
           ABBV     0.073326   0.056376   0.042995    0.032249  10.782336   
           ABT      0.033465   0.033071   0.039285    0.021639  18.250758   
           ACN      0.029801   0.021839   0.017126    0.010139  25.163335   
           ADBE     0.017918   0.023031   0.042073    0.026857  23.447905   

                        SMB        HML        RMW        CMA  cluster  \
Date       Ticker                                                       
2017-09-30 AAPL   -3.766800 -11.945149  18.515471   7.416116        2   
           ABBV    6.575230   0.132594   1.436639   2.463194        3   
           ABT    -3.315076 -10.978257   5.828022  19.639424        2   
           ACN    -3.063859  -5.825993   6.367886   1.512977        2   
           ADBE   -6.014644  -3.127997  -3.335185 -15.626848        2   

                                  role  TrendScore  RiskScore  QualityScore  \
Date       Ticker                                                             
2017-09-30 AAPL     Stables de qualité    0.053024   0.024639      5.720527   
           ABBV    Gagnants dynamiques    0.090537   0.038322     -5.437034   
           ABT      Stables de qualité    0.053024   0.024639      5.720527   
           ACN      Stables de qualité    0.053024   0.024639      5.720527   
           ADBE     Stables de qualité    0.053024   0.024639      5.720527   

                   StressScore  
Date       Ticker               
2017-09-30 AAPL      -0.058681  
           ABBV      -0.101019  
           ABT       -0.058681  
           ACN       -0.058681  
           ADBE      -0.058681

# 10) Construction de l’univers investissable (sélection top actions par rôle)

À chaque fin de mois, on ne garde pas toutes les actions :
- on sélectionne seulement les meilleures actions dans les rôles
  - **Gagnants dynamiques** (triés par `TrendScore`)
  - **Stables de qualité** (triés par `QualityScore`)

La sélection est :
- **Top fraction** : ex. top 50% (`TOP_FRAC = 0.50`)
- avec une contrainte de taille :
  - au moins `MIN_NAMES`
  - au plus `MAX_NAMES`

Résultat :
- un dictionnaire `universes` qui contient, pour chaque mois, la liste des tickers “winners” et “stables”.


In [15]:
# ============================================================
# 10) BUILD UNIVERSE (top selection within roles)
# ============================================================
def select_top_fraction(df_month, role_name, score_col, top_frac=TOP_FRAC, min_names=MIN_NAMES, max_names=MAX_NAMES):
    sub = df_month[df_month["role"] == role_name].copy().dropna(subset=[score_col])
    if sub.empty:
        return []
    sub = sub.sort_values(score_col, ascending=False)
    n = int(np.ceil(len(sub) * top_frac))
    n = max(min_names, n)
    n = min(max_names, n)
    n = min(len(sub), n)
    return sub.index[:n].tolist()

def build_universe_by_month(df_roles):
    universes = {}
    dates = df_roles.index.get_level_values("Date").unique().sort_values()
    for d in dates:
        m = df_roles.xs(d, level=0)
        winners = select_top_fraction(m, "Gagnants dynamiques", "TrendScore")
        stables = select_top_fraction(m, "Stables de qualité", "QualityScore")
        universes[d] = {"winners": winners, "stables": stables}
    return universes

universes = build_universe_by_month(df1_roles)

# 11) Construction du dataset LSTM (uniquement actions sélectionnées)

Objectif :
- Entraîner **un seul modèle LSTM global** (toutes actions confondues)
- Prédire le **return_1m du mois suivant** à partir des **SEQ_LEN derniers mois**

On ne prend pas toutes les actions :
- On utilise uniquement les tickers sélectionnés chaque mois dans l’univers :
  - `winners` + `stables`

Pour chaque couple (Date signal = d, Ticker = t) :
- **X** = matrice (SEQ_LEN × nb_features) des features sur les SEQ_LEN derniers mois jusqu’à *d*
- **y** = `return_1m` réalisé au mois suivant (*d+1M*)

Important (anti look-ahead) :
- les features utilisées dans X sont disponibles à la fin du mois *d*
- la cible y appartient au mois *d+1*


In [16]:
# ============================================================
# 11) LSTM DATASET CREATION (only SELECTED actions after clustering)
#     Target: next month return_1m
# ============================================================
# features for ML (include everything except labels)
ML_DROP = ["cluster","role"]  # keep scores + factors + techs
ml_features = [c for c in df1_roles.columns if c not in ML_DROP]

# We build a table for sequences per ticker, but only for months where ticker is selected
def make_lstm_dataset(df_roles, universes, seq_len=SEQ_LEN):
    rows = []
    dates = df_roles.index.get_level_values("Date").unique().sort_values()

    for d in dates:
        sel = list(dict.fromkeys(universes[d]["winners"] + universes[d]["stables"]))
        if not sel:
            continue

        # for each selected ticker at signal date d, we want:
        # X = previous seq_len months ending at d (inclusive), y = return_1m at next month (d+1M)
        next_d = (pd.Timestamp(d) + pd.offsets.MonthEnd(1)).normalize()
        for t in sel:
            # extract history for this ticker up to d
            g = df_roles.xs(t, level=1).sort_index()
            if d not in g.index or next_d not in g.index:
                continue
            idx_pos = g.index.get_loc(d)
            if idx_pos - (seq_len-1) < 0:
                continue

            window = g.iloc[idx_pos-(seq_len-1):idx_pos+1][ml_features].copy()

            # target: next month return_1m
            y = g.loc[next_d, "return_1m"]
            if window.isna().any().any() or pd.isna(y):
                continue

            rows.append({
                "Date": d,
                "Ticker": t,
                "X": window.values.astype(np.float32),
                "y": float(y)
            })

    if not rows:
        raise ValueError("Dataset LSTM vide. Essaie de réduire SEQ_LEN ou d'élargir la sélection.")

    df_ds = pd.DataFrame(rows)
    return df_ds

ds = make_lstm_dataset(df1_roles, universes, seq_len=SEQ_LEN)
print("Nb samples LSTM:", len(ds))
display(ds)

Nb samples LSTM: 2213


,Date,Ticker,X,y
0,2018-08-31,AAPL,"[[0.009763067, 45.768757, 0.018120449, -0.0602...",-0.008303
1,2018-08-31,ABT,"[[0.007494143, 63.664993, 0.0176685, 0.0475068...",0.097547
2,2018-08-31,ACN,"[[0.007929295, 56.25008, 0.015744483, 0.032961...",0.006684
3,2018-08-31,ADBE,"[[0.008798837, 47.932484, 0.016494658, -0.0385...",0.024439
4,2018-08-31,AMGN,"[[0.010598632, 61.740837, 0.02086195, 0.048826...",0.037436
...,...,...,...,...
2208,2023-08-31,GILD,"[[0.013322836, 45.065475, 0.027414028, -0.0170...",-0.002747
2209,2023-08-31,HD,"[[0.014641765, 43.467926, 0.031087864, -0.0432...",-0.084045
2210,2023-08-31,IBM,"[[0.01182889, 32.575394, 0.024826169, -0.07504...",-0.024450
2211,2023-08-31,ISRG,"[[0.018499702, 35.5106, 0.03479652, -0.0889472...",-0.070168


# 12) Découpage Train / Validation / Test (split temporel)

Comme on travaille sur des séries temporelles, on ne fait **pas** de split aléatoire.
On découpe chronologiquement :

- **Train** : ~70% des mois (apprentissage)
- **Validation** : ~15% (tuning / early stopping)
- **Test** : ~15% (évaluation finale)

Ainsi, on respecte la contrainte “on apprend sur le passé, on teste sur le futur”.


In [ ]:
# ============================================================
# 12) TRAIN / VAL / TEST SPLIT (time-based)
# ============================================================
ds["Date"] = pd.to_datetime(ds["Date"])
dates_sorted = ds["Date"].sort_values().unique()

cut1 = int(len(dates_sorted) * 0.70)
cut2 = int(len(dates_sorted) * 0.85)
train_dates = set(dates_sorted[:cut1])
val_dates   = set(dates_sorted[cut1:cut2])
test_dates  = set(dates_sorted[cut2:])

train = ds[ds["Date"].isin(train_dates)].copy()
val   = ds[ds["Date"].isin(val_dates)].copy()
test  = ds[ds["Date"].isin(test_dates)].copy()

X_train = np.stack(train["X"].values)
y_train = train["y"].values.astype(np.float32)

X_val = np.stack(val["X"].values)
y_val = val["y"].values.astype(np.float32)

X_test = np.stack(test["X"].values)
y_test = test["y"].values.astype(np.float32)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

Shapes: (1537, 12, 18) (311, 12, 18) (365, 12, 18)


# 13) Normalisation + entraînement du modèle LSTM

## 13.1 Normalisation
Avant d’entraîner un LSTM, on normalise les features :
- on **fit** le scaler uniquement sur le **train** (pour éviter toute fuite d’information)
- on applique la transformation sur train / val / test
- on standardise **chaque feature** sur l’ensemble des timesteps (reshape 2D)

## 13.2 Modèle LSTM
Architecture :
- 2 couches LSTM (64 puis 32 unités)
- Dropout pour réduire l’overfitting
- Dense final → prédiction d’un seul nombre : `return_1m` du mois suivant

On utilise :
- loss = MSE
- EarlyStopping sur `val_loss`


In [18]:
# ============================================================
# 13) NORMALIZE FEATURES (global scaler on training set)
#     We scale per feature across all timesteps.
# ============================================================
n_features = X_train.shape[-1]
scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, n_features)
X_val_2d   = X_val.reshape(-1, n_features)
X_test_2d  = X_test.reshape(-1, n_features)

scaler.fit(X_train_2d)

X_train_s = scaler.transform(X_train_2d).reshape(X_train.shape)
X_val_s   = scaler.transform(X_val_2d).reshape(X_val.shape)
X_test_s  = scaler.transform(X_test_2d).reshape(X_test.shape)

# ============================================================
# BUILD & TRAIN LSTM
# ============================================================
tf.keras.backend.clear_session()

model = models.Sequential([
    layers.Input(shape=(SEQ_LEN, n_features)),
    layers.Masking(mask_value=0.0),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss="mse"
)

cb = [
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

hist = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=LSTM_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
    callbacks=cb
)

# evaluation
test_pred = model.predict(X_test_s, verbose=0).reshape(-1)
mse = np.mean((test_pred - y_test)**2)
mae = np.mean(np.abs(test_pred - y_test))

print("Test MSE:", mse)
print(f"Test MAE: {mae:.6f}")



Epoch 1/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 188ms/step - loss: 0.0210 - val_loss: 0.0124
Epoch 2/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0126 - val_loss: 0.0111
Epoch 3/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.0098 - val_loss: 0.0109
Epoch 4/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.0089 - val_loss: 0.0114
Epoch 5/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - loss: 0.0081 - val_loss: 0.0113
Epoch 6/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 0.0079 - val_loss: 0.0109
Epoch 7/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 0.0078 - val_loss: 0.0107
Epoch 8/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 0.0077 - val_loss: 0.0110
Epoch 9/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.0079 - val_loss: 0.0110
Epoch 10/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 0.0078 - val_loss: 0.0104
Epoch 11/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.0077 - val_loss: 0.0102
Epoch 12/30
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 0.0073 - val_loss: 0.0103

# 14) Prédire le rendement du mois suivant (Expected Return)

Une fois le LSTM entraîné, on prédit `return_1m` du mois suivant pour chaque observation (Date, Ticker) présente dans notre dataset LSTM.

Résultat :
- `pred_df` : table indexée `(Date, Ticker)` contenant `pred_return_next`
- Ces prédictions seront utilisées comme **expected returns** (`mu_pred`) dans la construction du portefeuille.


In [19]:
# ============================================================
# 14) PREDICT NEXT MONTH RETURN FOR EACH (Date, Ticker) sample
#     We store predictions aligned with "signal date" d
# ============================================================
X_all = np.stack(ds["X"].values)
X_all_s = scaler.transform(X_all.reshape(-1, n_features)).reshape(X_all.shape)
ds["pred_return_next"] = model.predict(X_all_s, verbose=0).reshape(-1)

pred_df = (ds[["Date","Ticker","pred_return_next"]]
           .set_index(["Date","Ticker"])
           .sort_index())

display(pred_df)

pred_return_next
Date       Ticker                  
2018-08-31 AAPL           -0.032755
           ABT            -0.006139
           ACN            -0.038392
           ADBE           -0.061268
           AMGN            0.006055
...                             ...
2023-08-31 HD              0.002805
           IBM             0.012479
           ISRG           -0.068107
           JPM            -0.002188
           VLO            -0.003632

[2213 rows x 1 columns]